In [2]:
from Bio import SeqIO

records = list(
    SeqIO.parse(
        "../data/raw/BindingDBTargetSequences.fa",
        "fasta"
    )
)

print(
    len(records)
)

11433


In [3]:
print(records[0].id)
print(records[0].description)

p1
p1 mol:protein length:376 Thymidine kinase


In [4]:
print(records[100].id)
print(records[100].description)

p151
p151 mol:protein length:99 HIV-1 Protease Mutant (Q7K/L33I/L63I) chain A


In [7]:
import pandas as pd
protein_df = pd.DataFrame({
    "Protein_ID": [r.id for r in records],
    "Description": [r.description for r in records],
    "Sequence": [str(r.seq) for r in records]
})

protein_df.head()

,Protein_ID,Description,Sequence
0,p1,p1 mol:protein length:376 Thymidine kinase,MASYPCHQHASAFDQAARSRGHNNRRTALRPRRQQKATEVRLEQKM...
1,p2,p2 mol:protein length:159 Streptavadin(N23A),DPSKDSKAQVSAAEAGITGTWYAQLGSTFIVTAGADGALTGTYESA...
2,p3,p3 mol:protein length:159 Streptavadin(Y43A),DPSKDSKAQVSAAEAGITGTWYNQLGSTFIVTAGADGALTGTAESA...
3,p4,p4 mol:protein length:159 Streptavadin(S27A),DPSKDSKAQVSAAEAGITGTWYNQLGATFIVTAGADGALTGTYESA...
4,p5,p5 mol:protein length:183 Streptavidin,MRKIVVAAIAVSLTTVSITASASADPSKDSKAQVSAAEAGITGTWY...


In [8]:
records[0]

SeqRecord(seq=Seq('MASYPCHQHASAFDQAARSRGHNNRRTALRPRRQQKATEVRLEQKMPTLLRVYI...EAN'), id='p1', name='p1', description='p1 mol:protein length:376 Thymidine kinase', dbxrefs=[])

create data frame

Sequence Length Distribution

In [9]:
protein_df["Length"] = (
    protein_df["Sequence"]
    .apply(len)
)

protein_df["Length"].describe()

count    11433.000000
mean       595.359311
std        653.545961
min         51.000000
25%        310.000000
50%        451.000000
75%        690.000000
max      34350.000000
Name: Length, dtype: float64

In [10]:
protein_df.to_csv(
    "../data/processed/protein_sequences.csv",
    index=False
)

In [11]:
def encode_protein(
    seq
):

    return [
        AA_VOCAB[a]
        for a in seq
        if a in AA_VOCAB
    ]

In [12]:
AA_VOCAB = {
    "A":0,
    "C":1,
    "D":2,
    "E":3,
    "F":4,
    "G":5,
    "H":6,
    "I":7,
    "K":8,
    "L":9,
    "M":10,
    "N":11,
    "P":12,
    "Q":13,
    "R":14,
    "S":15,
    "T":16,
    "V":17,
    "W":18,
    "Y":19
}

In [13]:
encode_protein(
    protein_df.iloc[0]["Sequence"][:20]
)

[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, 2, 13, 0, 0, 14, 15, 14]

In [14]:
protein_df["Length"] = (
    protein_df["Sequence"]
    .apply(len)
)

protein_df["Length"].describe()

count    11433.000000
mean       595.359311
std        653.545961
min         51.000000
25%        310.000000
50%        451.000000
75%        690.000000
max      34350.000000
Name: Length, dtype: float64

In [15]:
protein_df["Tokens"] = (
    protein_df["Sequence"]
    .apply(encode_protein)
)

protein_df.head()

,Protein_ID,Description,Sequence,Length,Tokens
0,p1,p1 mol:protein length:376 Thymidine kinase,MASYPCHQHASAFDQAARSRGHNNRRTALRPRRQQKATEVRLEQKM...,376,"[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, ..."
1,p2,p2 mol:protein length:159 Streptavadin(N23A),DPSKDSKAQVSAAEAGITGTWYAQLGSTFIVTAGADGALTGTYESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
2,p3,p3 mol:protein length:159 Streptavadin(Y43A),DPSKDSKAQVSAAEAGITGTWYNQLGSTFIVTAGADGALTGTAESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
3,p4,p4 mol:protein length:159 Streptavadin(S27A),DPSKDSKAQVSAAEAGITGTWYNQLGATFIVTAGADGALTGTYESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
4,p5,p5 mol:protein length:183 Streptavidin,MRKIVVAAIAVSLTTVSITASASADPSKDSKAQVSAAEAGITGTWY...,183,"[10, 14, 8, 7, 17, 17, 0, 0, 7, 0, 17, 15, 9, ..."


In [16]:
len(
    protein_df.iloc[0]["Tokens"]
)

376

In [17]:
protein_df.iloc[0]["Tokens"][:20]

[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, 2, 13, 0, 0, 14, 15, 14]

In [18]:
MAX_LEN = 512

def pad_sequence(tokens):

    tokens = tokens[:MAX_LEN]

    padding = [0] * (
        MAX_LEN - len(tokens)
    )

    return tokens + padding

protein_df["InputIDs"] = (
    protein_df["Tokens"]
    .apply(pad_sequence)
)

protein_df.head()

,Protein_ID,Description,Sequence,Length,Tokens,InputIDs
0,p1,p1 mol:protein length:376 Thymidine kinase,MASYPCHQHASAFDQAARSRGHNNRRTALRPRRQQKATEVRLEQKM...,376,"[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, ...","[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, ..."
1,p2,p2 mol:protein length:159 Streptavadin(N23A),DPSKDSKAQVSAAEAGITGTWYAQLGSTFIVTAGADGALTGTYESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ...","[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
2,p3,p3 mol:protein length:159 Streptavadin(Y43A),DPSKDSKAQVSAAEAGITGTWYNQLGSTFIVTAGADGALTGTAESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ...","[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
3,p4,p4 mol:protein length:159 Streptavadin(S27A),DPSKDSKAQVSAAEAGITGTWYNQLGATFIVTAGADGALTGTYESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ...","[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
4,p5,p5 mol:protein length:183 Streptavidin,MRKIVVAAIAVSLTTVSITASASADPSKDSKAQVSAAEAGITGTWY...,183,"[10, 14, 8, 7, 17, 17, 0, 0, 7, 0, 17, 15, 9, ...","[10, 14, 8, 7, 17, 17, 0, 0, 7, 0, 17, 15, 9, ..."


In [19]:
len(
    protein_df.iloc[0]["InputIDs"]
)

512

In [20]:
import torch
import torch.nn as nn

In [21]:
class ProteinEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(
            20,
            128
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

    def forward(
        self,
        x
    ):

        x = self.embedding(x)

        x = self.transformer(x)

        return x.mean(
            dim=1
        )

In [22]:
model = ProteinEncoder()

sample = torch.tensor(
    [
        protein_df.iloc[0]["InputIDs"]
    ]
)

output = model(sample)

print(
    output.shape
)

torch.Size([1, 128])


In [23]:
protein_df.to_csv(
    "../data/processed/protein_sequences.csv",
    index=False
)

In [24]:
protein_df["Target_Name"] = (
    protein_df["Description"]
    .str.replace(
        r"^p\d+\s+mol:protein\s+length:\d+\s*",
        "",
        regex=True
    )
)

protein_df[
    ["Protein_ID", "Target_Name"]
].head()

,Protein_ID,Target_Name
0,p1,Thymidine kinase
1,p2,Streptavadin(N23A)
2,p3,Streptavadin(Y43A)
3,p4,Streptavadin(S27A)
4,p5,Streptavidin


In [25]:
protein_df["Target_Name"].sample(20)

377                          Monoacylglycerol lipase ABHD2
8418                                 Zinc transporter ZIP3
191        ADP-ribosyl cyclase/cyclic ADP-ribose hydrolase
6817                       Acylamino-acid-releasing enzyme
8592     tRNA (guanosine(18)-2'-O)-methyltransferase TA...
4238           Serine/threonine-protein kinase Nek2 [C22A]
3289     Phosphatidylinositol 4,5-bisphosphate 3-kinase...
10556                            Ceramide transfer protein
420             Bromodomain-containing protein 9 [134-239]
344                             Protein kinase C beta type
4487      Gamma-aminobutyric acid receptor subunit alpha-3
7420                         Mannose-6-phosphate isomerase
6580                           Palmitoyltransferase ZDHHC7
2334              DNA-directed RNA polymerase subunit beta
4762                                Tubulin alpha-1A chain
7822                              Ephrin type-A receptor 6
8650                           Botulinum neurotoxin type

In [26]:
import pandas as pd

binding_df = pd.read_csv(
    "../data/processed/bindingdb_clean.csv"
)

print(binding_df.shape)
binding_df.head()

(22232, 4)


,Ligand SMILES,Target Name,Ki (nM),pKi
0,CCC(c1ccccc1)c1c(O)c2ccccc2oc1=O,Dimer of Gag-Pol polyprotein [489-587],1000.0,6.000000
1,CCC(c1ccccc1)c1c(O)cc(CCc2ccccc2)oc1=O,Dimer of Gag-Pol polyprotein [489-587],500.0,6.301030
2,CCC(Cc1ccccc1)c1cc(O)c(C(CC)c2ccccc2)c(=O)o1,Dimer of Gag-Pol polyprotein [489-587],38.0,7.420216
3,Oc1c2CCCCCCc2oc(=O)c1C(C1CC1)c1ccccc1,Dimer of Gag-Pol polyprotein [489-587],15.0,7.823909
4,CCC(Cc1ccccc1)c1cc(O)c(C(CC)c2ccccc2)c(=O)o1,Dimer of Gag-Pol polyprotein [514-612],32.0,7.494850


In [27]:
binding_df["Target Name"].sample(20)

12967                  Mitogen-activated protein kinase 14
4788                                 Coagulation factor IX
5001                                  Carbonic anhydrase 1
16270                                       Beta-lactamase
20363                       5-hydroxytryptamine receptor 6
6072                                     Serine protease 1
3138                              Interstitial collagenase
1242                                         Neuraminidase
21625                            Replicase polyprotein 1ab
2968                                  Carbonic anhydrase 1
10531                                 Bcl-2-like protein 1
4691                                  Coagulation factor X
3408                                  Carbonic anhydrase 2
16606    Probable nicotinate-nucleotide adenylyltransfe...
5510                                  Coagulation factor X
19526    Isoform C of Bromodomain-containing protein 4 ...
20602                                 Carbonic anhydrase

In [28]:
protein_targets = set(
    protein_df["Target_Name"]
)

binding_targets = set(
    binding_df["Target Name"]
)

matches = protein_targets.intersection(
    binding_targets
)

print("Protein Targets:", len(protein_targets))
print("Binding Targets:", len(binding_targets))
print("Exact Matches:", len(matches))

Protein Targets: 8257
Binding Targets: 798
Exact Matches: 672


In [29]:
merged_df = binding_df.merge(
    protein_df,
    left_on="Target Name",
    right_on="Target_Name",
    how="inner"
)

print(merged_df.shape)

(98805, 11)


In [30]:
merged_df[
    [
        "Ligand SMILES",
        "Target Name",
        "pKi"
    ]
].head()

,Ligand SMILES,Target Name,pKi
0,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759
1,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759
2,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759
3,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759
4,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759


In [31]:
merged_df.columns

Index(['Ligand SMILES', 'Target Name', 'Ki (nM)', 'pKi', 'Protein_ID',
       'Description', 'Sequence', 'Length', 'Tokens', 'InputIDs',
       'Target_Name'],
      dtype='str')

In [32]:
merged_df.to_csv(
    "../data/processed/drug_protein_dataset.csv",
    index=False
)

In [33]:
merged_df[
    [
        "Ligand SMILES",
        "Target Name",
        "pKi"
    ]
].head()

,Ligand SMILES,Target Name,pKi
0,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759
1,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759
2,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759
3,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759
4,CN(C)CCCN1c2ccccc2Sc3c1cc(cc3)Cl,5-hydroxytryptamine receptor 7,6.318759


In [34]:
merged_df["pKi"].describe()



count    98805.000000
mean         6.510129
std          1.594588
min         -0.000000
25%          5.494850
50%          6.537602
75%          7.602060
max         12.698970
Name: pKi, dtype: float64

In [35]:
merged_df["Target Name"].nunique()

672

In [36]:
import pandas as pd

binding_df = pd.read_csv(
    "../data/processed/bindingdb_clean.csv"
)

protein_df = pd.read_csv(
    "../data/processed/protein_sequences.csv"
)

In [37]:
protein_df["Target_Name"] = (
    protein_df["Description"]
    .str.replace(
        r"^p\d+\s+mol:protein\s+length:\d+\s*",
        "",
        regex=True
    )
)

In [38]:
merged_df = binding_df.merge(
    protein_df,
    left_on="Target Name",
    right_on="Target_Name",
    how="inner"
)

print(merged_df.shape)

(98805, 11)


In [39]:
merged_df.to_csv(
    "../data/processed/drug_protein_dataset.csv",
    index=False
)

In [40]:
df = pd.read_csv(
    "../data/processed/drug_protein_dataset.csv"
)

print(df.shape)
print(df.columns)

(98805, 11)
Index(['Ligand SMILES', 'Target Name', 'Ki (nM)', 'pKi', 'Protein_ID',
       'Description', 'Sequence', 'Length', 'Tokens', 'InputIDs',
       'Target_Name'],
      dtype='str')


In [41]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(train_df.shape)
print(test_df.shape)

(79044, 11)
(19761, 11)


In [42]:
AA_VOCAB = {
    "A":0,
    "C":1,
    "D":2,
    "E":3,
    "F":4,
    "G":5,
    "H":6,
    "I":7,
    "K":8,
    "L":9,
    "M":10,
    "N":11,
    "P":12,
    "Q":13,
    "R":14,
    "S":15,
    "T":16,
    "V":17,
    "W":18,
    "Y":19
}

In [43]:
MAX_LEN = 512

def encode_protein(seq):

    tokens = [
        AA_VOCAB[a]
        for a in seq
        if a in AA_VOCAB
    ]

    tokens = tokens[:MAX_LEN]

    tokens += [0] * (
        MAX_LEN - len(tokens)
    )

    return tokens

In [44]:
sample_df = train_df.sample(
    5000,
    random_state=42
).reset_index(drop=True)

sample_df["ProteinTokens"] = (
    sample_df["Sequence"]
    .apply(encode_protein)
)

sample_df.head()

,Ligand SMILES,Target Name,Ki (nM),pKi,Protein_ID,Description,Sequence,Length,Tokens,InputIDs,Target_Name,ProteinTokens
0,CC(C)C[C@H](NC(=O)c1c[nH]c2ccccc12)C(=O)N[C@@H...,Replicase polyprotein 1ab,680.0,6.167491,p781,p781 mol:protein length:6729 Replicase polypro...,MFYNQVTLAVASDSEISGFGFAIPSVAVRTYSEAAAQGFQACRFVA...,6729,"[10, 4, 19, 11, 13, 17, 16, 9, 0, 17, 0, 15, 2...","[10, 4, 19, 11, 13, 17, 16, 9, 0, 17, 0, 15, 2...",Replicase polyprotein 1ab,"[10, 4, 19, 11, 13, 17, 16, 9, 0, 17, 0, 15, 2..."
1,Fc1ccc(cc1)C(=O)Oc1cc(=O)oc2c3cccc4CCn(c34)c(=...,Macrophage migration inhibitory factor,190.0,6.721246,p5931,p5931 mol:protein length:115 Macrophage migrat...,MPMFIVNTNVPRASVPEGFLSELTQQLAQATGKPAQYIAVHVVPDQ...,115,"[10, 12, 10, 4, 7, 17, 11, 16, 11, 17, 12, 14,...","[10, 12, 10, 4, 7, 17, 11, 16, 11, 17, 12, 14,...",Macrophage migration inhibitory factor,"[10, 12, 10, 4, 7, 17, 11, 16, 11, 17, 12, 14,..."
2,CNC(=O)[C@@H](NC(=O)[C@H](OCc1ccccc1C(=O)Nc1cc...,HIV-1 protease,70.0,7.154902,p50007923,p50007923 mol:protein length:99 HIV-1 protease,PQVTLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKM...,99,"[12, 13, 17, 16, 9, 18, 13, 14, 12, 9, 17, 16,...","[12, 13, 17, 16, 9, 18, 13, 14, 12, 9, 17, 16,...",HIV-1 protease,"[12, 13, 17, 16, 9, 18, 13, 14, 12, 9, 17, 16,..."
3,CN1CCN(CC(=O)Nc2cc(nc(n2)-c2ccco2)-n2cccn2)CC1,Adenosine receptor A2a,2.7,8.568636,p49000161,p49000161 mol:protein length:409 Adenosine rec...,MSSSVYITVELVIAVLAILGNVLVCWAVWINSNLQNVTNYFVVSLA...,409,"[10, 15, 15, 15, 17, 19, 7, 16, 17, 3, 9, 17, ...","[10, 15, 15, 15, 17, 19, 7, 16, 17, 3, 9, 17, ...",Adenosine receptor A2a,"[10, 15, 15, 15, 17, 19, 7, 16, 17, 3, 9, 17, ..."
4,COc1ccc(C(=O)NCC2CCC(CN)CC2)cc1OCCc1ccc(Cl)cc1Cl,Coagulation factor X,9345.0,5.029421,p1789,p1789 mol:protein length:482 Coagulation factor X,MESPVRLSLLYVVLASLLLPGRSVFINRERANNVLQRIRRANSFFE...,482,"[10, 3, 15, 12, 17, 14, 9, 15, 9, 9, 19, 17, 1...","[10, 3, 15, 12, 17, 14, 9, 15, 9, 9, 19, 17, 1...",Coagulation factor X,"[10, 3, 15, 12, 17, 14, 9, 15, 9, 9, 19, 17, 1..."


In [45]:
len(sample_df.iloc[0]["ProteinTokens"])

512

In [46]:
sample_df.to_pickle("../data/processed/pharmagpt_train.pkl")

print(sample_df.shape)

(5000, 12)


In [47]:
import torch

from torch.utils.data import Dataset

from rdkit import Chem

from torch_geometric.data import Data

c:\Users\shiva\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\shiva\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [48]:
def smiles_to_graph(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    x = []

    for atom in mol.GetAtoms():

        x.append([
            atom.GetAtomicNum()
        ])

    edges = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edges.append([i, j])
        edges.append([j, i])

    if len(edges) == 0:
        return None

    return Data(
        x=torch.tensor(
            x,
            dtype=torch.float
        ),

        edge_index=torch.tensor(
            edges,
            dtype=torch.long
        ).t()
    )

In [49]:
class PharmaGPTDataset(Dataset):

    def __init__(self, df):

        self.df = df

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        graph = smiles_to_graph(
            row["Ligand SMILES"]
        )

        protein = torch.tensor(
            row["ProteinTokens"],
            dtype=torch.long
        )

        label = torch.tensor(
            row["pKi"],
            dtype=torch.float
        )

        return (
            graph,
            protein,
            label
        )

In [50]:
dataset = PharmaGPTDataset(
    sample_df
)

print(
    len(dataset)
)

5000


In [51]:
graph, protein, label = dataset[0]

print(graph)
print()

print(protein.shape)

print()

print(label)

Data(x=[39, 1], edge_index=[2, 86])

torch.Size([512])

tensor(6.1675)


In [52]:
from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)

import torch.nn as nn
import torch

In [53]:
class DrugEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(1, 64)

        self.conv2 = GCNConv(64, 128)

    def forward(
        self,
        x,
        edge_index,
        batch
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = torch.relu(x)

        return global_mean_pool(
            x,
            batch
        )

In [54]:
class ProteinEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(20,128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

    def forward(self,x):

        x = self.embedding(x)

        x = self.transformer(x)

        return x.mean(dim=1)

In [55]:
class PharmaGPT(nn.Module):

    def __init__(self):

        super().__init__()

        self.drug_encoder = DrugEncoder()

        self.protein_encoder = ProteinEncoder()

        self.head = nn.Sequential(

            nn.Linear(256,128),

            nn.ReLU(),

            nn.Linear(128,1)
        )

    def forward(
    self,
    graph,
    protein,
    batch
):

        drug_emb = self.drug_encoder(
        graph.x,
        graph.edge_index,
        batch
    )

        protein_emb = self.protein_encoder(
        protein
    )

        x = torch.cat(
        [
            drug_emb,
            protein_emb
        ],
        dim=1
    )

        return self.head(x)

In [56]:
model = PharmaGPT()

print(model)

PharmaGPT(
  (drug_encoder): DrugEncoder(
    (conv1): GCNConv(1, 64)
    (conv2): GCNConv(64, 128)
  )
  (protein_encoder): ProteinEncoder(
    (embedding): Embedding(20, 128)
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (head): Sequential(
    (0): Linear(in_features=256,

In [57]:
graph, protein, label = dataset[0]

protein = protein.unsqueeze(0)

print(protein.shape)

torch.Size([1, 512])


In [58]:
batch = torch.zeros(
    graph.num_nodes,
    dtype=torch.long
)

In [59]:
drug_emb = model.drug_encoder(
    graph.x,
    graph.edge_index,
    batch
)

print(drug_emb.shape)

torch.Size([1, 128])


In [60]:
protein_emb = model.protein_encoder(
    protein
)

print(protein_emb.shape)

torch.Size([1, 128])


In [61]:
prediction = model(
    graph,
    protein,
    batch
)

print(prediction.shape)
print(prediction)

torch.Size([1, 1])
tensor([[-0.0898]], grad_fn=<AddmmBackward0>)


In [62]:
model = PharmaGPT()

In [63]:
graph, protein, label = dataset[0]

protein = protein.unsqueeze(0)

batch = torch.zeros(
    graph.num_nodes,
    dtype=torch.long
)

prediction = model(
    graph,
    protein,
    batch
)

print(prediction.shape)
print(prediction)

torch.Size([1, 1])
tensor([[0.1346]], grad_fn=<AddmmBackward0>)
